# IEEE Big Data Cup 2026 Shared Baseline Notebook

This notebook is the team's shared starting point for the AI Emulation Challenge. It is designed to be completed collaboratively in Google Colab and stored in the team's GitHub repository.

We will first create a simple, reproducible baseline. We are not trying to build the final model in this notebook today. We are creating one pipeline that everyone can understand, run, and improve.

## 1. Project Objective

Our objective is to build a machine-learning emulator of a theory-based ecosystem model.

The model should use environmental inputs and the initial ecosystem state to forecast ecosystem carbon variables over multiple years.

The first milestone is a working baseline that can:

- load the data
- identify inputs and targets
- define validation
- train a simple CPU-friendly model
- evaluate predictions
- eventually forecast autoregressively
- create a valid submission

A baseline is a reference point. It lets us determine whether future models and features actually improve the results.

In [ ]:
PROJECT_NAME = "IEEE Big Data Cup 2026 AI Emulation Challenge"
TEAM_GOAL = (
    "Build a reproducible CPU-friendly baseline for long-horizon ecosystem forecasting."
)

print(PROJECT_NAME)
print(TEAM_GOAL)

## 2. Imports

This section contains the libraries needed throughout the notebook.

We are intentionally beginning with lightweight tools that can run on ordinary computers and Colab CPU. Individual experiments may add libraries later, but the shared baseline should remain easy to reproduce.

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Imports complete.")

## 3. Load the Data

The shared notebook should run in Google Colab. The team can upload the files directly or mount them through Google Drive.

Recommended folder:

/content/data/

Expected file types may include CSV or Parquet. Confirm the exact names using the official competition files and tutorial.

In [ ]:
# Update this path if the team uses Google Drive.
DATA_DIR = Path("/content/data")

# Leave these as None for automatic detection.
TRAIN_FILE = None
TEST_FILE = None
SAMPLE_SUBMISSION_FILE = None

SUPPORTED_SUFFIXES = {".csv", ".parquet", ".feather"}

def read_table(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".feather":
        return pd.read_feather(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")

def choose_file(files, keyword):
    matches = [p for p in files if keyword.lower() in p.name.lower()]
    return matches[0] if matches else None

data_files = []
if DATA_DIR.exists():
    data_files = sorted(
        p for p in DATA_DIR.rglob("*")
        if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES
    )

print("Candidate files:")
for path in data_files:
    print(" -", path)

if TRAIN_FILE is None:
    TRAIN_FILE = choose_file(data_files, "train")
if TEST_FILE is None:
    TEST_FILE = choose_file(data_files, "test")
if SAMPLE_SUBMISSION_FILE is None:
    SAMPLE_SUBMISSION_FILE = choose_file(data_files, "sample")

print("\nTRAIN_FILE:", TRAIN_FILE)
print("TEST_FILE:", TEST_FILE)
print("SAMPLE_SUBMISSION_FILE:", SAMPLE_SUBMISSION_FILE)

if TRAIN_FILE is None or TEST_FILE is None:
    raise FileNotFoundError(
        "Set TRAIN_FILE and TEST_FILE manually after locating the competition files."
    )

train_df = read_table(TRAIN_FILE)
test_df = read_table(TEST_FILE)

print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head())

## 4. Understand the Data Shape

Before building a model, the team must document what the rows and columns mean.

Answer these questions:

- What does one row represent?
- Which columns identify a site, scenario, forest-age condition, or year?
- Does the training data contain one-step transitions or complete sequences?
- What is the forecast horizon?
- How are the sequences ordered?

This section should contain observations from the official tutorial, not guesses.

In [ ]:
print("Training columns:")
for i, column in enumerate(train_df.columns, start=1):
    print(f"{i:3}. {column}")

print("\nTrain-only columns:")
print(sorted(set(train_df.columns) - set(test_df.columns)))

print("\nTest-only columns:")
print(sorted(set(test_df.columns) - set(train_df.columns)))

print("\nData types:")
display(train_df.dtypes.to_frame("dtype"))

print("\nTraining shape:", train_df.shape)
print("Testing shape:", test_df.shape)

## 5. Identify Input Features and Targets

Input features are the information available to the model when it makes a prediction.

Targets are the ecosystem variables that the model is trying to predict.

Train-only columns are only a clue. Confirm the target columns using the official competition documentation and starter notebook.

Also separate identifier and metadata columns from actual predictive features.

In [ ]:
# Fill these in after reviewing the official tutorial.
TARGET_COLUMNS = []
ID_COLUMNS = []
TIME_COLUMNS = []

candidate_target_columns = sorted(set(train_df.columns) - set(test_df.columns))

print("Candidate train-only columns:")
print(candidate_target_columns)

if not TARGET_COLUMNS:
    print("\nCHECKPOINT: Fill TARGET_COLUMNS before training.")
else:
    missing_targets = [c for c in TARGET_COLUMNS if c not in train_df.columns]
    if missing_targets:
        raise KeyError(f"Target columns not found: {missing_targets}")

    feature_columns = [
        c for c in test_df.columns
        if c not in ID_COLUMNS
    ]

    print("Selected targets:", TARGET_COLUMNS)
    print("Number of features:", len(feature_columns))
    print("First features:", feature_columns[:20])
    display(train_df[TARGET_COLUMNS].describe().T)

## 6. Check Missing Values

Before training, inspect missing values, infinite values, and duplicate rows.

For each important column, the team should decide whether to:

- remove rows
- impute values
- use a competition-provided preprocessing method
- leave the values unchanged if the model supports them

Do not silently ignore missing values. Record the decision in the README and experiment log.

In [ ]:
print("Missing-value fractions in training data:")
missing_summary = train_df.isna().mean().sort_values(ascending=False)
display(missing_summary.head(25).to_frame("missing_fraction"))

print("Duplicate training rows:", train_df.duplicated().sum())
print("Duplicate testing rows:", test_df.duplicated().sum())

numeric_columns = train_df.select_dtypes(include=np.number).columns
infinite_counts = np.isinf(train_df[numeric_columns]).sum().sort_values(ascending=False)
print("Columns containing infinite values:")
display(infinite_counts[infinite_counts > 0].to_frame("infinite_count"))

## 7. Define the Validation Split

Validation estimates how well the model will perform on unseen data.

Our final validation should resemble the hidden future forecasting task. A random row split may leak information if rows from the same site or sequence appear in both training and validation.

First, review the official tutorial to determine whether it already provides a validation split. If not, create a time-respecting or sequence-respecting split.

The temporary split below is only a smoke test to make sure the notebook runs. It is not automatically the final scientific validation.

In [ ]:
USE_TEMPORARY_SPLIT = True
VALIDATION_FRACTION = 0.20

def make_temporary_split(df, fraction=0.20):
    cutoff = int(len(df) * (1 - fraction))
    return df.iloc[:cutoff].copy(), df.iloc[cutoff:].copy()

if USE_TEMPORARY_SPLIT:
    train_part, valid_part = make_temporary_split(
        train_df,
        fraction=VALIDATION_FRACTION,
    )
    print("Temporary training rows:", len(train_part))
    print("Temporary validation rows:", len(valid_part))
    print("Replace this with the official/time-respecting split.")
else:
    raise NotImplementedError(
        "Implement the official validation split after reviewing the tutorial."
    )

## 8. Create a Simple Baseline

A baseline is a simple reference model that gives us a first score.

We will begin with a small Random Forest because it:

- can model nonlinear relationships
- works with tabular data
- is relatively approachable for beginners
- can run on CPU
- gives us a reference before we try more complicated methods

This code assumes the official data can be represented as a tabular one-step prediction problem. If the organizer notebook uses a different sequence format, adapt this section to the official baseline instead of forcing the data into this format.

In [ ]:
def prepare_tabular_data(df, feature_columns, target_columns):
    X = df[feature_columns].copy()
    y = df[target_columns].copy()

    non_numeric = X.select_dtypes(exclude=np.number).columns.tolist()
    if non_numeric:
        raise TypeError(
            "Non-numeric features found. Review and encode them intentionally: "
            f"{non_numeric}"
        )

    X = X.replace([np.inf, -np.inf], np.nan)
    y = y.replace([np.inf, -np.inf], np.nan)

    keep_rows = X.notna().all(axis=1) & y.notna().all(axis=1)
    return X.loc[keep_rows], y.loc[keep_rows]

if TARGET_COLUMNS:
    feature_columns = [c for c in test_df.columns if c not in ID_COLUMNS]

    X_train, y_train = prepare_tabular_data(
        train_part,
        feature_columns,
        TARGET_COLUMNS,
    )
    X_valid, y_valid = prepare_tabular_data(
        valid_part,
        feature_columns,
        TARGET_COLUMNS,
    )

    baseline_model = RandomForestRegressor(
        n_estimators=50,
        max_depth=12,
        max_samples=0.5,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )

    start_time = time.time()
    baseline_model.fit(X_train, y_train)
    training_seconds = time.time() - start_time

    valid_predictions = baseline_model.predict(X_valid)

    print(f"Training time: {training_seconds:.1f} seconds")
    print("Validation prediction shape:", valid_predictions.shape)
else:
    print("Fill TARGET_COLUMNS before running the baseline.")

## 9. Evaluate Predictions

First, calculate standard regression metrics such as RMSE and MAE.

Then add the exact competition metrics from the official starter notebook, including:

- cumulative error
- year-to-year delta error
- any target-specific or scenario-specific metrics

Evaluate each target separately as well as overall performance. A model may perform well on one target and poorly on another.

In [ ]:
def calculate_basic_metrics(y_true, y_pred):
    return {
        "rmse_overall": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae_overall": float(mean_absolute_error(y_true, y_pred)),
    }

if TARGET_COLUMNS:
    print(calculate_basic_metrics(y_valid, valid_predictions))

    target_metrics = []
    for index, target in enumerate(TARGET_COLUMNS):
        target_metrics.append({
            "target": target,
            "rmse": np.sqrt(
                mean_squared_error(y_valid.iloc[:, index], valid_predictions[:, index])
            ),
            "mae": mean_absolute_error(
                y_valid.iloc[:, index],
                valid_predictions[:, index],
            ),
        })

    metrics_df = pd.DataFrame(target_metrics)
    display(metrics_df)

    plt.figure(figsize=(9, 4))
    plt.plot(y_valid.iloc[:, 0].to_numpy()[:200], label="Actual")
    plt.plot(valid_predictions[:200, 0], label="Predicted")
    plt.title(f"Baseline predictions: {TARGET_COLUMNS[0]}")
    plt.xlabel("Validation row")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Run the baseline first.")

## 10. Autoregressive Forecasting

This is the most important competition-specific section.

The model should forecast multiple years by feeding its own prediction into the next step:

1. Predict Year 1.
2. Use the predicted Year 1 state to predict Year 2.
3. Use the predicted Year 2 state to predict Year 3.
4. Continue through the full forecast horizon.

Before implementing this, identify from the official tutorial:

- site or sequence identifier
- year or order column
- initial state columns
- environmental driver columns
- target columns
- forecast horizon
- test-row ordering
- which prediction becomes the next year's state

Never use hidden true future target values during rollout.

In [ ]:
def autoregressive_forecast(
    model,
    sequence_df,
    driver_columns,
    state_columns,
    horizon,
):
    # Complete this function after confirming the official sequence structure.
    # The implementation must use each prediction as the next state.
    raise NotImplementedError(
        "Implement the official autoregressive rollout before reporting "
        "long-horizon results."
    )

print("Autoregressive forecasting is a required team checkpoint.")

## 11. Create a Submission File

Only create the final submission after confirming the official format.

Use the sample submission as the template. Do not guess:

- row order
- column names
- identifier columns
- prediction ordering
- whether predictions need reshaping or flattening

First create a small test submission and verify that Kaggle accepts the structure.

In [ ]:
if SAMPLE_SUBMISSION_FILE is not None:
    sample_submission = read_table(SAMPLE_SUBMISSION_FILE)
    print("Sample submission shape:", sample_submission.shape)
    print("Submission columns:", list(sample_submission.columns))
    display(sample_submission.head())
else:
    sample_submission = None
    print("Locate the official sample submission before continuing.")

In [ ]:
def create_submission_template(sample_submission, predictions):
    if sample_submission is None:
        raise ValueError("Load the official sample submission first.")

    if len(sample_submission) != len(predictions):
        raise ValueError(
            "The number of prediction rows does not match the sample submission."
        )

    # Complete this mapping after reviewing the official submission format.
    raise NotImplementedError(
        "Map predictions to the official submission columns, then save the file "
        "inside the submissions folder."
    )

print("Submission formatting is a required team checkpoint.")

## 12. Conclusion

At the end of each experiment, summarize:

- model used
- features used
- validation approach
- one-step metrics
- full-horizon metrics
- training time
- Kaggle score, if submitted
- limitations
- recommended next experiment

Record the result in experiment_log.csv before merging changes into the shared GitHub branch.

### Team workflow checklist

- [ ] The notebook opens and runs in Colab.
- [ ] The data structure is documented.
- [ ] Targets and features are confirmed from the official tutorial.
- [ ] Missing-value decisions are documented.
- [ ] Validation avoids future leakage.
- [ ] The CPU-friendly baseline runs.
- [ ] One-step metrics are recorded.
- [ ] Autoregressive evaluation uses predictions, not hidden true targets.
- [ ] The sample submission format is confirmed.
- [ ] The experiment log is updated.
- [ ] The README explains how to run the notebook.

In [ ]:
# Example experiment-log row.
experiment_record = {
    "experiment_name": "random_forest_baseline",
    "date": pd.Timestamp.now().strftime("%Y-%m-%d"),
    "model": "RandomForestRegressor",
    "features": "confirm after official tutorial",
    "validation_split": "confirm official/time-respecting split",
    "random_seed": RANDOM_SEED,
    "one_step_rmse": None,
    "full_horizon_rmse": None,
    "mae": None,
    "cumulative_error": None,
    "training_time_seconds": None,
    "kaggle_score": None,
    "conclusion": "Initial reference model.",
}

pd.DataFrame([experiment_record])